# Train one stage (helpful)
Train Qwen2.5-1.5B-Instruct with DPO + 4-bit QLoRA on the 2,000 helpful training pairs, then check it learned.

**Needs a GPU:** in Colab, Runtime → Change runtime type → **L4** (or A100).

What we want to find out:
1. Does training work at all? (loss goes down from 0.69)
2. How long does one stage take, and how much GPU memory? (this tells us if 1,024 tokens is affordable)
3. Did it learn? (the model should prefer the chosen answers on **helpful val** pairs it has never seen)

In [ ]:
import os, sys

if os.path.exists("/content"):   # on Colab: get the latest code + data from GitHub
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    !pip install -q peft bitsandbytes
    REPO = "/content/mfr-dpo"
else:
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")
import importlib, mfr_data, mfr_dpo
importlib.reload(mfr_data)
importlib.reload(mfr_dpo)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

In [ ]:
splits = mfr_data.load_splits(f"{REPO}/data")
train = splits["helpful"]["train"]
print(len(train), "helpful training pairs")

## 1. Quick test (100 pairs, a few minutes)
Just checks that everything runs and the loss starts going down from 0.69 (= the model has no preference yet).
If you run out of GPU memory, set `micro_batch=1`.

In [ ]:
model, tokenizer = mfr_dpo.load_model()
model.print_trainable_parameters()

quick = mfr_dpo.train_stage(model, tokenizer, train.head(100), log_every=1)

## 2. Full stage (2,000 pairs)
We reload the model first so this run starts from scratch, not from the quick test.

In [ ]:
del model
torch.cuda.empty_cache()
model, tokenizer = mfr_dpo.load_model()

BETA, LR = 0.1, 5e-5   # starting guesses; we tune them in the pilot
history = mfr_dpo.train_stage(model, tokenizer, train, beta=BETA, lr=LR)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history["step"], history["loss"].rolling(10, min_periods=1).mean(), color="#2a78d6", linewidth=2)
ax.axhline(0.693, color="#6b6a66", linestyle="--", linewidth=1)
ax.text(history["step"].max(), 0.70, "no preference (0.69)", ha="right", va="bottom", fontsize=9, color="#6b6a66")
ax.set_title("Training loss (smoothed)", loc="left", fontsize=11)
ax.set_xlabel("optimizer step")
ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Did it learn?
For every **val** pair we compute the margin: how much more the trained model prefers the chosen answer
than the untrained model did. Before training every margin is exactly 0.

* **helpful**: should clearly go up (accuracy well above 50%). This is what we just trained on.
* **safe / quality**: not trained yet. This is the "before" picture for measuring forgetting later.

In [ ]:
import pandas as pd

results = {}
for name in ["helpful", "safe", "quality"]:
    m = mfr_dpo.score_margins(model, tokenizer, splits[name]["val"], beta=BETA)
    results[name] = {"accuracy (margin > 0)": f"{100 * (m > 0).mean():.0f}%",
                     "mean margin": round(m.mean(), 4)}
pd.DataFrame(results)

## 4. Save the adapter (optional)
The adapter is small (about 70 MB). Colab deletes everything when the session ends, so save it to the team Drive
if you want to keep it. In VS Code, mount Drive first with Command Palette → "Colab: Mount Google Drive to Server".

In [ ]:
SAVE = False
DRIVE_DIR = "/content/drive/MyDrive/mfr-dpo"   # change to the team folder's name in your Drive

if SAVE:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained(f"{DRIVE_DIR}/adapters/helpful_one_stage")
    history.to_csv(f"{DRIVE_DIR}/adapters/helpful_one_stage/history.csv", index=False)